# Stage 4b — Where does the view bias come from?

Question: E1 scores healthy AP films higher than healthy PA films (view leakage 0.785 on test).
Does that come from cues **outside** the lungs (H1: text, cables, borders) or from the
**lung region** itself (H2: projection physics, including the smaller AP lung silhouette)?

**Fixed before running** (rules from `docs/PROYECTO.md` v0.2, section 6, plus implementation details):
- Validation set only, images with suspect lung masks excluded. E1 = mean logit of the 3 seeds.
- Masks dilated by 3 px. Primary test: replaced region set to black (−1024). Robustness: replaced region blurred (σ = 8 px), which keeps coarse intensity but removes text and cables.
- Decision on E1, lungs-only (black): leakage ≤ 0.65 → H1; ≥ 0.75 → H2; in between → both.
  If within-view AUC drops by more than 0.05, the test is inconclusive and E2 decides.
- Explanations: only Grad-CAM and HiResCAM@denseblock3 (Integrated Gradients failed the sanity check in 4a).

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from src.data import REPO_ROOT, find_data_root, LABELS_FILE, CLASSES_FILE
from src.dataset import CachedCXRDataset
from src.masks import MaskedCXRDataset, load_masks
from src.metrics import bootstrap_auc
from src.models import OpacityNet, load_xrv_densenet, predict_logits
from src.train import load_trained
from src.utils import get_device

DATA, CKPT = REPO_ROOT / "data", REPO_ROOT / "checkpoints"
RESULTS, FIG = REPO_ROOT / "results", REPO_ROOT / "docs" / "figures"
cache_path = DATA / "processed" / "images_224.npy"
mask_path = DATA / "processed" / "lung_masks_224.npy"
device = get_device()
seeds = [0, 1, 2]

root = find_data_root()
cls = pd.read_csv(root / CLASSES_FILE).drop_duplicates("patientId")[["patientId", "class"]]
stats = pd.read_csv(mask_path.with_suffix(".stats.csv"))[["patientId", "area_frac", "suspect"]]
splits = pd.read_csv(DATA / "splits.csv")
if "class" not in splits.columns:          # splits.csv may already carry the detailed class
    splits = splits.merge(cls, on="patientId")
splits = splits.merge(stats, on="patientId")val = splits[splits.split == "val"]
vs = val[~val.suspect].reset_index(drop=True)   # mask-based analyses exclude suspect masks
print(pd.crosstab([val["class"], val.view], val.suspect, margins=True))

models = {"E0": OpacityNet(load_xrv_densenet()).to(device).eval()}
for s in seeds:
    models[f"E1_seed{s}"] = load_trained(CKPT / f"e1_seed{s}.pt", device)

KeyError: 'class'

## 1. Decisive test: inference with only the lungs (or only the outside) visible

Five input conditions × four models (E0 and the three E1 seeds) on the validation set.
About 25–30 min on MPS.

In [ ]:
CONDITIONS = {
    "full": None,
    "lungs_only_black": ("lung", "black"),
    "lungs_only_blur": ("lung", "blur"),
    "outside_only_black": ("outside", "black"),
    "outside_only_blur": ("outside", "blur"),
}

# What the model sees in each condition (one Normal AP study)
example = vs[(vs["class"] == "Normal") & (vs.view == "AP")].patientId.iloc[:1]
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(2.4 * len(CONDITIONS), 2.6))
for ax, (cond, spec) in zip(axes, CONDITIONS.items()):
    ds = (CachedCXRDataset(cache_path, example) if spec is None
          else MaskedCXRDataset(cache_path, mask_path, example, mode=spec[0], fill=spec[1]))
    ax.imshow(ds[0][0], cmap="gray", vmin=-1024, vmax=1024)
    ax.set_title(cond, fontsize=8)
    ax.axis("off")
fig.savefig(FIG / "masking_conditions.png", dpi=120, bbox_inches="tight")
plt.show()

for cond, spec in CONDITIONS.items():
    ds = (CachedCXRDataset(cache_path, vs.patientId) if spec is None
          else MaskedCXRDataset(cache_path, mask_path, vs.patientId, mode=spec[0], fill=spec[1]))
    loader = DataLoader(ds, batch_size=64, num_workers=0)
    for name, model in models.items():
        vs[f"{cond}|{name}"] = predict_logits(model, loader, device)[:, 0]
    vs[f"{cond}|E1"] = vs[[f"{cond}|E1_seed{s}" for s in seeds]].mean(axis=1)
    print("done:", cond)
vs.to_csv(DATA / "predictions" / "val_masking_test.csv", index=False)

In [ ]:
normal = vs[vs["class"] == "Normal"]
is_ap_normal = (normal.view == "AP").astype(int)
rows = []
for cond in CONDITIONS:
    for name in ["E0", "E1"]:
        col = f"{cond}|{name}"
        leak, lo, hi = bootstrap_auc(is_ap_normal, normal[col])
        rows.append({
            "condition": cond, "model": name,
            "leakage": leak, "ci_low": lo, "ci_high": hi,
            "auc_AP": roc_auc_score(vs.target[vs.view == "AP"], vs[col][vs.view == "AP"]),
            "auc_PA": roc_auc_score(vs.target[vs.view == "PA"], vs[col][vs.view == "PA"]),
        })
masking = pd.DataFrame(rows)
print(masking.round(3).to_string(index=False))
masking.to_csv(RESULTS / "stage4_masking_test.csv", index=False)

# Reference: how much does the lung-mask area alone reveal the view among Normal studies?
print("\nmask area alone, AP vs PA among Normal:", round(roc_auc_score(is_ap_normal, -normal.area_frac), 3))

In [ ]:
def pick(cond, model="E1"):
    return masking[(masking.condition == cond) & (masking.model == model)].iloc[0]

for fill in ["black", "blur"]:
    full, lung = pick("full"), pick(f"lungs_only_{fill}")
    auc_drop = max(full.auc_AP - lung.auc_AP, full.auc_PA - lung.auc_PA)
    if auc_drop > 0.05:
        verdict = "INCONCLUSIVE: lung-only input is too far out of distribution; E2 decides."
    elif lung.leakage <= 0.65:
        verdict = "H1 supported: the view bias comes mainly from outside the lungs."
    elif lung.leakage >= 0.75:
        verdict = "H2 supported: the view bias survives with only the lung region visible."
    else:
        verdict = "Intermediate: both regions contribute."
    tag = "PRIMARY" if fill == "black" else "robustness"
    print(f"[{tag}, {fill}] E1 leakage full {full.leakage:.3f} -> lungs only {lung.leakage:.3f}; "
          f"max within-view AUC drop {auc_drop:.3f}\n  {verdict}")

## 2. Where do the explanations point?

Grad-CAM and HiResCAM@denseblock3 for E0 and E1 (seed 0) on every non-suspect validation image.
Per image: **LAF** (share of positive attribution inside the lungs) and **LAF ratio** (LAF divided
by lung area; 1 = no preference for the lungs). On positives, the **pointing game** (does the
peak fall inside a radiologist box, ±15 px) against the chance of a random point in the lungs.
About 15–20 min on MPS.

In [ ]:
from src.explain import cam
from src.xai_metrics import attribution_summary, box_masks, mean_ci

METHODS = {
    "Grad-CAM": lambda m, x: cam(m, x, method="gradcam"),
    "HiResCAM@db3": lambda m, x: cam(m, x, layer=m.features.denseblock3, method="hirescam"),
}
lung = load_masks(mask_path, vs.patientId)
boxes = box_masks(pd.read_csv(root / LABELS_FILE), vs.patientId)
groups = (vs["class"] + "|" + vs.view).to_numpy()
ds = CachedCXRDataset(cache_path, vs.patientId)

per_image, mean_maps = [], {}
for mname in ["E0", "E1_seed0"]:
    for meth, fn in METHODS.items():
        df, means = attribution_summary(models[mname], fn, ds, lung, groups, device, boxes=boxes)
        per_image.append(df.assign(model=mname, method=meth))
        mean_maps[(mname, meth)] = means
        print("done:", mname, meth)
per_image = pd.concat(per_image).merge(vs[["class", "view"]], left_on="i", right_index=True)
per_image.to_csv(DATA / "predictions" / "val_attribution_per_image.csv", index=False)

In [ ]:
laf_rows = []
for (mname, meth, c, v), d in per_image.groupby(["model", "method", "class", "view"]):
    m, lo, hi = mean_ci(d.laf_ratio)
    laf_rows.append({"model": mname, "method": meth, "class": c, "view": v, "n": len(d),
                     "laf": d.laf.mean(), "laf_ratio": m, "ci_low": lo, "ci_high": hi})
laf = pd.DataFrame(laf_rows)
print(laf.round(3).to_string(index=False))
laf.to_csv(RESULTS / "stage4_laf.csv", index=False)

pointing = (per_image.dropna(subset=["hit"])
            .groupby(["model", "method", "view"])
            .agg(n=("hit", "size"), pointing=("hit", "mean"), chance=("chance", "mean"))
            .round(3))
print(pointing)
pointing.to_csv(RESULTS / "stage4_pointing_game.csv")

In [ ]:
# Mean attribution maps: Normal AP vs Normal PA, and their difference (red = more attribution in AP)
mean_lung = {g: lung[groups == g].mean(0) for g in ["Normal|AP", "Normal|PA"]}
fig, axes = plt.subplots(len(mean_maps), 3, figsize=(8, 2.8 * len(mean_maps)))
for r, key in enumerate(mean_maps):
    ap, pa = mean_maps[key]["Normal|AP"], mean_maps[key]["Normal|PA"]
    vmax, lim = max(ap.max(), pa.max()), np.abs(ap - pa).max()
    panels = [(ap, "Normal AP", "Normal|AP", dict(cmap="jet", vmin=0, vmax=vmax)),
              (pa, "Normal PA", "Normal|PA", dict(cmap="jet", vmin=0, vmax=vmax)),
              (ap - pa, "AP − PA", "Normal|AP", dict(cmap="bwr", vmin=-lim, vmax=lim))]
    for c, (img, title, g, kw) in enumerate(panels):
        ax = axes[r, c]
        ax.imshow(img, **kw)
        ax.contour(mean_lung[g], levels=[0.5], colors="k", linewidths=0.8)
        ax.set_title(f"{key[0]} · {key[1]}\n{title}", fontsize=8)
        ax.axis("off")
fig.tight_layout()
fig.savefig(FIG / "mean_attribution_normal_ap_pa.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Individual examples: the 16 Normal AP studies E1 scores highest vs 16 random Normal PA studies
from src.viz import show_grid

normal_vs = vs[vs["class"] == "Normal"]
examples = {"normal_ap_top": normal_vs[normal_vs.view == "AP"].nlargest(16, "full|E1"),
            "normal_pa_random": normal_vs[normal_vs.view == "PA"].sample(16, random_state=0)}
for name, df in examples.items():
    dsx = CachedCXRDataset(cache_path, df.patientId)
    x = torch.stack([dsx[i] for i in range(len(dsx))]).to(device)
    heat = METHODS["Grad-CAM"](models["E1_seed0"], x)
    fig = show_grid(x[:, 0].cpu().numpy(), load_masks(mask_path, df.patientId), heat,
                    titles=[f"p = {1 / (1 + np.exp(-z)):.2f}" for z in df["full|E1"]])
    fig.suptitle(f"E1 Grad-CAM: {name}", y=1.01)
    fig.savefig(FIG / f"gradcam_{name}.png", dpi=110, bbox_inches="tight")
    plt.show()

## 3. Does the representation encode the view, and does the head use it?

Linear probe (logistic regression) for AP vs PA on the 1024-d embedding, trained on 6,000 training
images and evaluated on validation. A high AUC only says the information is **available**.
To see whether the head **uses** it: cosine between the probe direction and the classifier
weights, and the mean logit gap between Normal AP and Normal PA, for E0 vs E1. *Exploratory.*
About 10 min on MPS.

In [ ]:
from src.probe import embeddings, view_probe, view_direction, cosine

train_sub = splits[splits.split == "train"].sample(6000, random_state=0)
train_loader = DataLoader(CachedCXRDataset(cache_path, train_sub.patientId), batch_size=64, num_workers=0)
val_loader = DataLoader(CachedCXRDataset(cache_path, vs.patientId), batch_size=64, num_workers=0)
y_train = (train_sub.view == "AP").to_numpy()
y_val = (vs.view == "AP").to_numpy()
is_normal = (vs["class"] == "Normal").to_numpy()

rows = []
for mname in ["E0", "E1_seed0"]:
    m = models[mname]
    e_tr, e_va = embeddings(m, train_loader, device), embeddings(m, val_loader, device)
    probe, auc_all = view_probe(e_tr, y_train, e_va, y_val)
    auc_normal = roc_auc_score(y_val[is_normal], probe.decision_function(e_va[is_normal]))
    w_head = m.classifier.weight.detach().cpu().numpy()[0]
    delta = e_va[is_normal & y_val].mean(0) - e_va[is_normal & ~y_val].mean(0)
    rows.append({"model": mname, "probe_auc_all": auc_all, "probe_auc_normal": auc_normal,
                 "cos(head, view direction)": cosine(w_head, view_direction(probe)),
                 "logit gap Normal AP-PA": float(w_head @ delta)})
probe_table = pd.DataFrame(rows)
print(probe_table.round(3).to_string(index=False))
probe_table.to_csv(RESULTS / "stage4_view_probe.csv", index=False)

## Conclusions (fill in after running)

- Decisive test (E1, lungs only, black): leakage ... → verdict ...
- Robustness (blur): ...
- Outside only: ...
- LAF / mean maps: ...
- Probe: ...